# 🧩 Notebook 3 · Hotel Management — Polymorphism & Patterns

Notebooks 1 and 2 built a clean booking engine. This notebook adds the three
**OOD moves you'll meet in every interview**:

1. **Strategy** — swap pricing rules (standard → seasonal → member) without
   touching `Hotel` or `Reservation`.
2. **Factory** — build rooms from config so the caller never instantiates
   classes by hand.
3. **Observer** — notify email/SMS systems when a booking happens, with zero
   coupling.

We'll keep the same domain (`Room`, `Guest`, `Reservation`) so concepts stack
on top of Notebook 2.

> **Why patterns?** They are *names* for shapes we'd invent anyway. Learning
> the names lets teammates point at code and agree what it is in one word.


## 🛠️ Setup

```bash
cd 07-object-oriented-design/hotel-management
uv sync
```

Select the `.venv` kernel (top-right of the notebook). If it's missing:
`Cmd+Shift+P` → **Reload Window**.


## 1️⃣ Shared domain (same as Notebook 2)

Copied here so this notebook runs on its own. Nothing new yet.


In [ ]:
from dataclasses import dataclass
from datetime import date
from enum import Enum
from abc import ABC, abstractmethod
import itertools

class RoomType(Enum):
    SINGLE = 100
    DOUBLE = 150
    SUITE  = 300

@dataclass(frozen=True)
class Room:
    number: int
    type: RoomType

@dataclass(frozen=True)
class Guest:
    id: str
    name: str
    is_member: bool = False

@dataclass
class Reservation:
    id: int
    room: Room
    guest: Guest
    check_in: date
    check_out: date

    @property
    def nights(self) -> int:
        return (self.check_out - self.check_in).days


## 2️⃣ Attempt 0 — pricing via `if/elif` (bad)

The naive approach: stuff every pricing rule into one method. It works for two
rules, then collapses. Every new rule means editing a class that already
works — a textbook violation of the **Open/Closed Principle**
("open for extension, closed for modification").


In [ ]:
def price_bad(res: Reservation, season: str) -> int:
    base = res.nights * res.room.type.value
    # 🔴 more rules -> more branches -> more bugs
    if season == "HIGH":
        base = int(base * 1.30)
    elif season == "LOW":
        base = int(base * 0.85)
    if res.guest.is_member:
        base = int(base * 0.90)
    return base

ada = Guest("G1", "Ada", is_member=True)
r = Reservation(1, Room(201, RoomType.DOUBLE), ada, date(2026,5,1), date(2026,5,4))
print("bad price (HIGH+member):", price_bad(r, "HIGH"))


### What's wrong?
- 🔴 Every new pricing rule edits `price_bad` — regression risk grows forever.
- 🔴 Rules can't be combined or reordered cleanly.
- 🔴 Impossible to unit-test one rule in isolation.


## 3️⃣ Best — the **Strategy pattern**

A *strategy* is just "an interface for one decision". We declare
`PricingStrategy` as an abstract base class (ABC) and give each rule its own
tiny class. `Hotel` accepts any strategy object — it doesn't care which.

This is the **Open/Closed Principle** in action: add a new rule by adding a
new class; never modify existing ones.


In [ ]:
class PricingStrategy(ABC):
    @abstractmethod
    def price(self, res: Reservation) -> int: ...

class StandardPricing(PricingStrategy):
    """Room rate x nights. The default."""
    def price(self, res):
        return res.nights * res.room.type.value

class SeasonalPricing(PricingStrategy):
    """Wrap another strategy and scale it."""
    def __init__(self, inner: PricingStrategy, multiplier: float):
        self.inner, self.multiplier = inner, multiplier
    def price(self, res):
        return int(self.inner.price(res) * self.multiplier)

class MemberDiscount(PricingStrategy):
    """10% off for members -- otherwise defers to the inner strategy."""
    def __init__(self, inner: PricingStrategy, pct: float = 0.10):
        self.inner, self.pct = inner, pct
    def price(self, res):
        total = self.inner.price(res)
        return int(total * (1 - self.pct)) if res.guest.is_member else total

# Composing strategies = Decorator pattern on top of Strategy.
high_season_member = MemberDiscount(SeasonalPricing(StandardPricing(), 1.30))
low_season_public  = SeasonalPricing(StandardPricing(), 0.85)

print("HIGH + member :", high_season_member.price(r))
print("LOW  + public :", low_season_public.price(r))


### Why this is better
- 🟢 Each rule is a tiny class with one responsibility — trivially testable.
- 🟢 New rule? Add a class. **Zero** edits to existing code.
- 🟢 Rules compose: `MemberDiscount(SeasonalPricing(StandardPricing(), 1.3))`
  reads like English.
- 🟢 `Hotel` depends on the *abstraction* `PricingStrategy`, not concrete
  classes — that's the **Dependency Inversion Principle**.


## 4️⃣ **Factory pattern** — stop `new`-ing classes by hand

Creating rooms from config is a chore, and callers shouldn't know which
concrete class to build. A **factory** centralizes construction so the rest
of the code stays blissfully ignorant.


In [ ]:
class RoomFactory:
    """Build rooms from simple config dicts. One place to change room creation."""
    _TYPE_MAP = {"S": RoomType.SINGLE, "D": RoomType.DOUBLE, "X": RoomType.SUITE}

    @classmethod
    def from_code(cls, number: int, code: str) -> Room:
        if code not in cls._TYPE_MAP:
            raise ValueError(f"unknown room code {code!r}; use one of {list(cls._TYPE_MAP)}")
        return Room(number, cls._TYPE_MAP[code])

    @classmethod
    def from_config(cls, config):
        return [cls.from_code(c["num"], c["code"]) for c in config]

rooms = RoomFactory.from_config([
    {"num": 101, "code": "S"},
    {"num": 201, "code": "D"},
    {"num": 301, "code": "X"},
])
for r_ in rooms:
    print(r_)


### When a factory pays off
- You load rooms from JSON / a DB / a CSV. Parsing lives in **one** place.
- You might swap `Room` for `PremiumRoom` later — callers never change.
- You want to validate inputs (bad `code` → clear error) at the boundary.


## 5️⃣ **Observer pattern** — notifications without coupling

When a booking is created we may need to:
- email the guest a confirmation,
- SMS the front desk,
- update an analytics dashboard.

We do **not** want `Hotel.reserve` to import an email library. Instead,
`Hotel` publishes a `booking_created` event and anyone interested subscribes.


In [ ]:
class BookingObserver(ABC):
    @abstractmethod
    def on_booking(self, res: Reservation) -> None: ...

class EmailNotifier(BookingObserver):
    def on_booking(self, res):
        print(f"  email to {res.guest.name}: reservation #{res.id} confirmed")

class SmsNotifier(BookingObserver):
    def on_booking(self, res):
        print(f"  SMS to front-desk: room {res.room.number} booked")

class Analytics(BookingObserver):
    def __init__(self):
        self.count = 0
    def on_booking(self, res):
        self.count += 1
        print(f"  analytics: total bookings so far = {self.count}")


## 6️⃣ Putting it all together — a pattern-aware `Hotel`

`Hotel` now accepts a `PricingStrategy` (Strategy + DIP) and a list of
`BookingObserver` (Observer). Construction of rooms stays in `RoomFactory`
(Factory). Each pattern solves exactly one problem.


In [ ]:
class Hotel:
    def __init__(self, rooms, pricing, observers=None):
        self._rid = itertools.count(1)          # per-instance, see notebook 2
        self.rooms = {r.number: r for r in rooms}
        self.reservations = []
        self.pricing = pricing
        self.observers = list(observers or [])

    def subscribe(self, obs):
        self.observers.append(obs)

    def _publish(self, res):
        for obs in self.observers:
            obs.on_booking(res)

    def _overlaps(self, room_number, ci, co):
        return any(
            r.room.number == room_number and ci < r.check_out and r.check_in < co
            for r in self.reservations
        )

    def reserve(self, room_number, guest, ci, co):
        if co <= ci:
            raise ValueError("check_out must be after check_in")
        if self._overlaps(room_number, ci, co):
            raise RuntimeError(f"room {room_number} not available")
        res = Reservation(next(self._rid), self.rooms[room_number], guest, ci, co)
        self.reservations.append(res)
        self._publish(res)
        return res

    def quote(self, res):
        return self.pricing.price(res)

In [ ]:
# Build the hotel once -- then swap parts freely.
rooms   = RoomFactory.from_config([{"num":101,"code":"S"},{"num":201,"code":"D"},{"num":301,"code":"X"}])
pricing = MemberDiscount(SeasonalPricing(StandardPricing(), 1.30))   # HIGH season, members 10% off
analytics = Analytics()
hotel = Hotel(rooms, pricing, observers=[EmailNotifier(), SmsNotifier(), analytics])

ada   = Guest("G1", "Ada",   is_member=True)
grace = Guest("G2", "Grace", is_member=False)

print("-- booking 1 --")
r1 = hotel.reserve(201, ada,   date(2026,7,1), date(2026,7,4))
print(f"  quote for Ada   : ${hotel.quote(r1)}")

print("-- booking 2 --")
r2 = hotel.reserve(301, grace, date(2026,7,2), date(2026,7,5))
print(f"  quote for Grace : ${hotel.quote(r2)}")

print(f"\nanalytics saw {analytics.count} bookings")


## 7️⃣ Swap the strategy at runtime — proof that it's decoupled

Same hotel, same rooms, different pricing — no class edits.


In [ ]:
hotel.pricing = StandardPricing()           # turn off season + member rules
print("standard quote for Ada :", hotel.quote(r1))

hotel.pricing = SeasonalPricing(StandardPricing(), 0.85)   # LOW season flash sale
print("low-season for Ada     :", hotel.quote(r1))


## 7️⃣b Verify the patterns — assertions, not vibes

A pattern is only *implemented* if you can demonstrate the property it exists to give you.
So we assert the property, not the class name:

- **Strategy** → swapping the strategy changes the price, and `Hotel` needs zero edits.
- **Decorator** → wrapping composes, and the order of wrapping is visible in the number.
- **Factory** → bad input is rejected *at construction*, in one place.
- **Observer** → a subscriber added *after* the hotel was built still receives events, and
  `Hotel` never learns what any observer does.

In [ ]:
# --- Strategy: same reservation, different strategy, different price -------
res = Reservation(99, Room(201, RoomType.DOUBLE), Guest("G9", "Zoe", is_member=True),
                  date(2026,5,1), date(2026,5,4))       # 3 nights x $150 = $450
assert res.nights == 3
assert StandardPricing().price(res) == 450

# Decorator: each wrapper defers to the one it wraps, so they compose.
assert SeasonalPricing(StandardPricing(), 1.30).price(res) == 585        # 450 * 1.3
assert MemberDiscount(SeasonalPricing(StandardPricing(), 1.30)).price(res) == 526  # 585 * 0.9

# A non-member walks straight through MemberDiscount untouched — the wrapper
# decides, the wrapped strategy never knows it was wrapped.
public = Reservation(98, res.room, Guest("G8", "Pat", is_member=False), res.check_in, res.check_out)
assert MemberDiscount(StandardPricing()).price(public) == StandardPricing().price(public)

# --- Open/Closed: a brand-new rule, zero edits to anything above ----------
class LongStayDiscount(PricingStrategy):
    """$20 off per night for stays of 3+ nights."""
    def __init__(self, inner): self.inner = inner
    def price(self, r):
        base = self.inner.price(r)
        return base - 20 * r.nights if r.nights >= 3 else base

assert LongStayDiscount(StandardPricing()).price(res) == 390             # 450 - 60

# --- Factory: construction is validated in exactly one place -------------
assert RoomFactory.from_code(7, "X").type is RoomType.SUITE
try:
    RoomFactory.from_code(7, "Z")
    raise AssertionError("unknown room code must be rejected at the factory")
except ValueError:
    pass

# --- Observer: late subscribers work; the subject stays ignorant ----------
class RecordingObserver(BookingObserver):
    def __init__(self): self.seen = []
    def on_booking(self, r): self.seen.append(r.id)

h = Hotel(RoomFactory.from_config([{"num": 101, "code": "S"}]), StandardPricing())
spy = RecordingObserver()
h.subscribe(spy)                                     # subscribed AFTER construction
b1 = h.reserve(101, Guest("G1", "Ada"), date(2026,7,1), date(2026,7,3))
b2 = h.reserve(101, Guest("G2", "Bob"), date(2026,7,3), date(2026,7,5))
assert spy.seen == [b1.id, b2.id],       "every booking reaches every subscriber, in order"

# Adding a second observer requires no change to Hotel.
spy2 = RecordingObserver(); h.subscribe(spy2)
b3 = h.reserve(101, Guest("G3", "Cy"), date(2026,7,5), date(2026,7,6))
assert spy.seen[-1] == spy2.seen[-1] == b3.id
assert spy2.seen == [b3.id],             "a new observer only sees events after it subscribed"

# --- Dependency Inversion: Hotel accepts ANY PricingStrategy -------------
h.pricing = LongStayDiscount(StandardPricing())      # a class Hotel has never heard of
assert h.quote(b1) == StandardPricing().price(b1)    # 2 nights -> discount does not apply

print("all pattern guarantees hold ✅")

## 8️⃣ Pattern cheat-sheet (hotel edition)

| Pattern   | Answers the question…                 | Where we used it                     |
|-----------|----------------------------------------|--------------------------------------|
| Strategy  | "How do I swap **one decision**?"      | Pricing rules (`PricingStrategy`)    |
| Decorator | "How do I **stack** small behaviours?" | `MemberDiscount(SeasonalPricing(…))` |
| Factory   | "Who **builds** my objects?"           | `RoomFactory.from_config`            |
| Observer  | "Who cares when **X happens**?"        | `EmailNotifier`, `SmsNotifier`       |

All four keep `Hotel` small and testable — its job stays *orchestration*.

## 🧠 Exercises
1. **Tax strategy.** Add `TaxStrategy` that wraps any pricing strategy and
   adds a country-specific VAT. Bonus: make it composable with
   `MemberDiscount`.
2. **Logging observer.** Write a `FileLogger(BookingObserver)` that appends
   one line per booking to `bookings.log`. Does `Hotel` change? (It shouldn't.)
3. **Guest factory.** Add `GuestFactory.from_email(email)` that derives
   `id`/`name` and decides `is_member` from a whitelist set.
4. **Strategy vs. inheritance.** Rewrite `StandardPricing` as a `Room` method
   (`room.price(nights)`). What breaks when we add seasonal pricing?
   *(That's why Strategy wins for volatile rules.)*

### Key takeaways
- **Strategy** = one interface, many swappable implementations.
- **Decorator** = wrap a strategy to add behaviour without subclassing.
- **Factory** = centralise construction; callers stay ignorant of concrete types.
- **Observer** = publish events; subscribers plug in without coupling.
- The **SOLID** principles (Open/Closed, Dependency Inversion, Single-
  Responsibility) aren't trivia — they fall out naturally once you use these
  patterns on purpose.
